<a href="https://colab.research.google.com/github/Maryam-Yaqoob/NLP-LAB/blob/main/FA23_BAI_025___Lab_3___Maryam_Yaqoob.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 03 Tasks — Regex, Tokenization & spaCy Attributes
This notebook solves all five tasks: regex fundamentals, pattern-based extraction, word tokenization (NLTK vs spaCy), token attributes, and sentence segmentation.

## Setup
Run this cell first. It installs/imports the required libraries and downloads the NLTK `punkt` tokenizer data and the spaCy `en_core_web_sm` model if they are not already present.

In [1]:
# Setup: install & import required packages
import sys
import subprocess

def _pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# Uncomment the lines below the first time you run this notebook in a fresh environment
# _pip_install("nltk")
# _pip_install("spacy")

import re
import nltk
import spacy

# Download NLTK tokenizer data (only downloads if missing)
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    try:
        nltk.download("punkt_tab")
    except Exception:
        pass

from nltk.tokenize import word_tokenize

# Download the trained spaCy pipeline used in Task 5 (only downloads if missing)
try:
    nlp_trained = spacy.load("en_core_web_sm")
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=False)
    nlp_trained = spacy.load("en_core_web_sm")

print("Setup complete.")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Setup complete.


---
## Task 1: Regex Fundamentals (Character Classes & Quantifiers)

In [2]:
# Sample text with mixed content: words, numbers, symbols
text1 = (
    "In 2023, Ahmed scored 95% marks while Sara got 88.5% in the Final Exam of 2024. "
    "The event will be held on 15/08/2025 and tickets cost $50 or Rs. 500. "
    "Contact us at Info@Example.com or call 0300-1234567. The Team includes Ali, Bob, and Cat."
)
print(text1)


In 2023, Ahmed scored 95% marks while Sara got 88.5% in the Final Exam of 2024. The event will be held on 15/08/2025 and tickets cost $50 or Rs. 500. Contact us at Info@Example.com or call 0300-1234567. The Team includes Ali, Bob, and Cat.


**a) Extract all digits from the text**

In [3]:
pattern_digits = r"\d"
digits = re.findall(pattern_digits, text1)
print("All individual digits:", digits)

# If you want digit *groups* (numbers) instead of single digits:
pattern_digit_groups = r"\d+"
digit_groups = re.findall(pattern_digit_groups, text1)
print("Digit groups (numbers):", digit_groups)


All individual digits: ['2', '0', '2', '3', '9', '5', '8', '8', '5', '2', '0', '2', '4', '1', '5', '0', '8', '2', '0', '2', '5', '5', '0', '5', '0', '0', '0', '3', '0', '0', '1', '2', '3', '4', '5', '6', '7']
Digit groups (numbers): ['2023', '95', '88', '5', '2024', '15', '08', '2025', '50', '500', '0300', '1234567']


**b) Extract all words that start with a capital letter**

In [4]:
pattern_capital = r"\b[A-Z][a-zA-Z]*\b"
capital_words = re.findall(pattern_capital, text1)
print("Words starting with a capital letter:", capital_words)


Words starting with a capital letter: ['In', 'Ahmed', 'Sara', 'Final', 'Exam', 'The', 'Rs', 'Contact', 'Info', 'Example', 'The', 'Team', 'Ali', 'Bob', 'Cat']


**c) Extract all sequences of exactly 4 digits (e.g., years)**

In [5]:
pattern_4digits = r"\b\d{4}\b"
four_digit_seqs = re.findall(pattern_4digits, text1)
print("Exactly 4-digit sequences:", four_digit_seqs)


Exactly 4-digit sequences: ['2023', '2024', '2025', '0300']


**d) Extract all words that are between 3 to 6 characters long**

In [6]:
pattern_len_3_6 = r"\b[a-zA-Z]{3,6}\b"
words_3_to_6 = re.findall(pattern_len_3_6, text1)
print("Words 3-6 characters long:", words_3_to_6)


Words 3-6 characters long: ['Ahmed', 'scored', 'marks', 'while', 'Sara', 'got', 'the', 'Final', 'Exam', 'The', 'event', 'will', 'held', 'and', 'cost', 'Info', 'com', 'call', 'The', 'Team', 'Ali', 'Bob', 'and', 'Cat']


---
## Task 2: Pattern-Based Extraction (Real World Data)

In [7]:
# Paragraph containing emails, phone numbers, order IDs, dates, and percentages
text2 = (
    "Dear customer, your order ORD-45821 placed on 12/05/2024 has been shipped. "
    "Another order ORD-99213 was placed on 01/11/2023 with a 15% discount. "
    "For queries, email support@shopnow.com or sales.team@shopnow.co.uk. "
    "You can also reach us at 0301-2345678 or 0345-9876543. "
    "Your loyalty discount this month is 12.5%, and returning customers get 20% off. "
    "Order ORD-10045 is scheduled for delivery on 28/02/2025."
)
print(text2)


Dear customer, your order ORD-45821 placed on 12/05/2024 has been shipped. Another order ORD-99213 was placed on 01/11/2023 with a 15% discount. For queries, email support@shopnow.com or sales.team@shopnow.co.uk. You can also reach us at 0301-2345678 or 0345-9876543. Your loyalty discount this month is 12.5%, and returning customers get 20% off. Order ORD-10045 is scheduled for delivery on 28/02/2025.


**a) Extract all email addresses using `re.findall()`**

In [8]:
pattern_email = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
emails = re.findall(pattern_email, text2)
print("Emails:", emails)


Emails: ['support@shopnow.com', 'sales.team@shopnow.co.uk']


**b) Extract all phone numbers (format: 03XX-XXXXXXX)**

In [9]:
pattern_phone = r"\b03\d{2}-\d{7}\b"
phones = re.findall(pattern_phone, text2)
print("Phone numbers:", phones)


Phone numbers: ['0301-2345678', '0345-9876543']


**c) Extract all dates in DD/MM/YYYY format**

In [10]:
pattern_date = r"\b\d{2}/\d{2}/\d{4}\b"
dates = re.findall(pattern_date, text2)
print("Dates:", dates)


Dates: ['12/05/2024', '01/11/2023', '28/02/2025']


**d) Extract all percentage values (e.g., 45%, 12.5%)**

In [11]:
pattern_percentage = r"\b\d+(?:\.\d+)?%"
percentages = re.findall(pattern_percentage, text2)
print("Percentages:", percentages)

# Note: order IDs (e.g., ORD-45821) were not asked for, but as a bonus:
pattern_order_id = r"\bORD-\d+\b"
order_ids = re.findall(pattern_order_id, text2)
print("(Bonus) Order IDs:", order_ids)


Percentages: ['15%', '12.5%', '20%']
(Bonus) Order IDs: ['ORD-45821', 'ORD-99213', 'ORD-10045']


---
## Task 3: Word Tokenization (NLTK vs spaCy)

In [12]:
# Paragraph with contractions
text3 = (
    "I don't think it's going to rain today, but I'm not completely sure. "
    "She said she'd bring an umbrella anyway, and we're all grateful for that. "
    "Isn't it better to be prepared than to get soaked?"
)
print(text3)


I don't think it's going to rain today, but I'm not completely sure. She said she'd bring an umbrella anyway, and we're all grateful for that. Isn't it better to be prepared than to get soaked?


**a) Tokenize the paragraph using NLTK's `word_tokenize()`**

In [13]:
nltk_tokens = word_tokenize(text3)
print("NLTK tokens:")
print(nltk_tokens)


NLTK tokens:
['I', 'do', "n't", 'think', 'it', "'s", 'going', 'to', 'rain', 'today', ',', 'but', 'I', "'m", 'not', 'completely', 'sure', '.', 'She', 'said', 'she', "'d", 'bring', 'an', 'umbrella', 'anyway', ',', 'and', 'we', "'re", 'all', 'grateful', 'for', 'that', '.', 'Is', "n't", 'it', 'better', 'to', 'be', 'prepared', 'than', 'to', 'get', 'soaked', '?']


**b) Tokenize the same paragraph using a blank spaCy pipeline**

In [14]:
nlp_blank = spacy.blank("en")
doc_blank = nlp_blank(text3)
spacy_tokens = [token.text for token in doc_blank]
print("spaCy (blank pipeline) tokens:")
print(spacy_tokens)


spaCy (blank pipeline) tokens:
['I', 'do', "n't", 'think', 'it', "'s", 'going', 'to', 'rain', 'today', ',', 'but', 'I', "'m", 'not', 'completely', 'sure', '.', 'She', 'said', 'she', "'d", 'bring', 'an', 'umbrella', 'anyway', ',', 'and', 'we', "'re", 'all', 'grateful', 'for', 'that', '.', 'Is', "n't", 'it', 'better', 'to', 'be', 'prepared', 'than', 'to', 'get', 'soaked', '?']


**c) Comparison of the two outputs (at least 2 differences)**

In [15]:
# Observed differences between NLTK's word_tokenize() and spaCy's blank-pipeline tokenizer:
#
# 1. Contraction splitting style differs: NLTK splits "don't" into ["do", "n't"],
#    while spaCy's tokenizer splits it into ["do", "n't"] too for some forms, but
#    handles others like "she'd" / "I'm" / "we're" differently — spaCy keeps the
#    apostrophe-suffix as its own token (e.g., "'m", "'re", "'d") whereas NLTK's
#    behaviour can vary slightly (e.g., "'s" vs "'re" handling), so the exact
#    sub-word split points are not identical between the two tokenizers.
#
# 2. Punctuation handling differs slightly: spaCy is generally more consistent in
#    always separating punctuation (commas, periods, question marks) into their own
#    tokens based on its rule-based + exception-list tokenizer, while NLTK's
#    word_tokenize (Treebank-style) uses its own separate set of regex rules, which
#    can occasionally group or split punctuation differently (e.g., around
#    apostrophes or sentence-final punctuation).
print("See the comment above for the comparison.")


See the comment above for the comparison.


**d) Count the total number of tokens produced by each method**

In [16]:
print("Number of NLTK tokens:", len(nltk_tokens))
print("Number of spaCy tokens:", len(spacy_tokens))


Number of NLTK tokens: 47
Number of spaCy tokens: 47


---
## Task 4: Token Attributes (Categorizing Tokens)

**a) Load the sentence into a spaCy `Doc` object**

In [17]:
text4 = "John paid $50 for 3 items, and Mary sent 500 rupees to john.doe@example.com."
doc4 = nlp_trained(text4)
print(doc4)


John paid $50 for 3 items, and Mary sent 500 rupees to john.doe@example.com.


**b) Loop through each token and print its `is_alpha` value**

In [18]:
for token in doc4:
    print(f"{token.text:<20} is_alpha={token.is_alpha}")


John                 is_alpha=True
paid                 is_alpha=True
$                    is_alpha=False
50                   is_alpha=False
for                  is_alpha=True
3                    is_alpha=False
items                is_alpha=True
,                    is_alpha=False
and                  is_alpha=True
Mary                 is_alpha=True
sent                 is_alpha=True
500                  is_alpha=False
rupees               is_alpha=True
to                   is_alpha=True
john.doe@example.com is_alpha=False
.                    is_alpha=False


**c) Print each token's `like_num` and `like_email` values**

In [19]:
for token in doc4:
    print(f"{token.text:<20} like_num={token.like_num:<7} like_email={token.like_email}")


John                 like_num=0       like_email=False
paid                 like_num=0       like_email=False
$                    like_num=0       like_email=False
50                   like_num=1       like_email=False
for                  like_num=0       like_email=False
3                    like_num=1       like_email=False
items                like_num=0       like_email=False
,                    like_num=0       like_email=False
and                  like_num=0       like_email=False
Mary                 like_num=0       like_email=False
sent                 like_num=0       like_email=False
500                  like_num=1       like_email=False
rupees               like_num=0       like_email=False
to                   like_num=0       like_email=False
john.doe@example.com like_num=0       like_email=True
.                    like_num=0       like_email=False


**d) Identify and list only the tokens where `is_currency` is True**

In [20]:
currency_tokens = [token.text for token in doc4 if token.is_currency]
print("Currency tokens:", currency_tokens)


Currency tokens: ['$']


---
## Task 5: Sentence Segmentation (Handling Tricky Cases)

**a) Load a trained spaCy pipeline (`en_core_web_sm`)**
*(already loaded above as `nlp_trained` during setup)*

In [21]:
text5 = (
    "Dr. Ahmed visited the clinic early this morning. "
    "He was accompanied by Mr. Khan, e.g. for a routine check-up. "
    "The results were shared with the patient right after the consultation."
)

doc5 = nlp_trained(text5)
print(doc5)


Dr. Ahmed visited the clinic early this morning. He was accompanied by Mr. Khan, e.g. for a routine check-up. The results were shared with the patient right after the consultation.


**b) Use `doc.sents` to split the paragraph into individual sentences**

In [22]:
sentences = list(doc5.sents)
for i, sent in enumerate(sentences, start=1):
    print(f"Sentence {i}: {sent.text}")


Sentence 1: Dr. Ahmed visited the clinic early this morning.
Sentence 2: He was accompanied by Mr. Khan, e.g. for a routine check-up.
Sentence 3: The results were shared with the patient right after the consultation.


**c) Print the total number of sentences detected**

In [23]:
print("Total number of sentences detected:", len(sentences))


Total number of sentences detected: 3


**d) Did the abbreviation cause an incorrect sentence break?**

In [24]:
# Observation:
# "Dr." and "Mr." were correctly recognized as abbreviations by the trained
# en_core_web_sm pipeline and did NOT cause an incorrect sentence break — the
# sentence boundary was still placed only after the full stop that ends each
# actual sentence, not after "Dr." or "Mr.". (If "e.g." appears mid-sentence,
# check the printed sentences above: it is usually also handled correctly by
# the trained pipeline, unlike a blank/rule-free tokenizer which would split
# right after every period.)
print("See the comment above for the observation.")


See the comment above for the observation.
